In [1]:
# Inicia Spark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder \
    .appName("titanic-silver") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://titanic-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "admin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [3]:
# Lê do Bronze

In [4]:
df = spark.read.format("delta") \
    .load("s3a://titanic/lakehouse/bronze/titanic")

df.printSchema()
df.show(5)

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+-----+--------+
|        892|       0|     3|    Kelly, Mr. James|  male|34.5|    0|    0| 330911| 7.8292| NULL|       Q|
|        893|       1|     3|Wilkes, Mrs. Jame...|female|47.0|    1|    0| 363272|    7.0| NULL|       S|
|      

In [5]:
# limpeza e tipagem

In [6]:
# df_silver = df \
#     .withColumn("Age", when(col("Age").isNull(), 29.0).otherwise(col("Age"))) \
#     .withColumn("Embarked", when(col("Embarked").isNull(), "S").otherwise(col("Embarked"))) \
#     .withColumn("Survived", col("Survived").cast("boolean")) \
#     .drop("Cabin")


#  Registra o DataFrame como uma tabela temporária
df.createOrReplaceTempView("df_bronze")

df_silver = spark.sql("""
    SELECT 
        PassengerId,
        CAST(Survived AS INTEGER) AS Survived,
        Pclass,
        Name,
        Sex,
        COALESCE(Age, 29) AS Age,
        SibSp,
        Parch,
        Ticket,
        Fare,
        COALESCE(Embarked, 'S') AS Embarked
    FROM df_bronze
""")


df_silver.show(5)

+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch| Ticket|   Fare|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+--------+
|        892|       0|     3|    Kelly, Mr. James|  male|34.5|    0|    0| 330911| 7.8292|       Q|
|        893|       1|     3|Wilkes, Mrs. Jame...|female|47.0|    1|    0| 363272|    7.0|       S|
|        894|       0|     2|Myles, Mr. Thomas...|  male|62.0|    0|    0| 240276| 9.6875|       Q|
|        895|       0|     3|    Wirz, Mr. Albert|  male|27.0|    0|    0| 315154| 8.6625|       S|
|        896|       1|     3|Hirvonen, Mrs. Al...|female|22.0|    1|    1|3101298|12.2875|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+-------+-------+--------+
only showing top 5 rows



In [7]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .save("s3a://titanic/lakehouse/silver/titanic")

print("Silver gravado com sucesso")

Silver gravado com sucesso
